In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!nvidia-smi

Wed Apr 29 10:22:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers accelerate peft bitsandbytes huggingface_hub sentencepiece protobuf --upgrade-strategy only-if-needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 99.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-cloud-videointelligence 2.18.0 requires protobuf

In [4]:
from huggingface_hub import login, whoami

login()
print(whoami())

{'type': 'user', 'id': '69ef7ec3c0ae7c02319d7e8b', 'name': 'RajeevSK25', 'fullname': 'Rajeev Shyam Kumar', 'isPro': False, 'avatarUrl': '/avatars/add2c6523550ad126c132db4374e13c1.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Gemma Write', 'role': 'fineGrained', 'createdAt': '2026-04-28T15:47:24.635Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '69f0d57017102d0119436781', 'type': 'model', 'name': 'RajeevSK25/gemma4-e2b-seo-lora'}, 'permissions': ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '69ef7ec3c0ae7c02319d7e8b', 'type': 'user', 'name': 'RajeevSK25'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write']}]}}}}


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "google/gemma-4-E2B-it"
adapter_id = "RajeevSK25/gemma4-e2b-seo-lora-v4-400"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, adapter_id)

model.eval()

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

try:
    model.config.use_cache = True
except Exception:
    pass

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded Gemma base model + v4 SEO LoRA adapter")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/38.1M [00:00<?, ?B/s]

Loaded Gemma base model + v4 SEO LoRA adapter


In [8]:
import re
import torch

def normalize_headings(text):
    text = text.strip()
    replacements = {
        r"Diagnosis\s*:": "Diagnosis:",
        r"Evidence\s*:": "Evidence:",
        r"Priority\s*:": "Priority:",
        r"Fix\s*:": "Fix:",
        r"Next action\s*:": "Next action:",
        r"Next action\s*\n": "Next action:\n",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def clean_first_complete_answer(text):
    text = normalize_headings(text)

    stop_markers = [
        "\nsystem\n",
        "\nmodel\n",
        "\ntype\n",
        "\nevidence\n",
        "\npriority\n",
        "\nfix\n",
        "\nnext action\n",
        "<start_of_turn>model",
        "<end_of_turn>",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    second_diag = text.find("\nDiagnosis:", 5)
    if second_diag != -1:
        text = text[:second_diag].strip()

    return normalize_headings(text)


def has_all_headings(response):
    required = ["Diagnosis:", "Evidence:", "Priority:", "Fix:", "Next action:"]
    return all(h in response for h in required)


def generate_response(messages, max_new_tokens=220):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=6,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return clean_first_complete_answer(raw_response)


def seo_agent(user_prompt):
    system_prompt = """You are an evidence-led SEO and marketing assistant.

You must answer using exactly five headings:
Diagnosis:
Evidence:
Priority:
Fix:
Next action:

Hard rules:
- Each heading must be one short sentence only.
- Evidence must mention maximum 5 missing or provided evidence points.
- Do not create long comma-separated lists.
- Do not add extra sections.
- Stop after Next action.
- Do not mention paid ads, display ads, ad copy, ROAS, or non-organic channels unless the user provides paid-channel data.
- If data is missing, say what is missing but still complete all five headings.
- Prioritise by business impact."""

    forced_user_prompt = f"""{user_prompt}

Return exactly this format only:
Diagnosis: one short sentence.
Evidence: one short sentence with maximum 5 evidence points.
Priority: one short sentence.
Fix: one short sentence.
Next action: one short sentence.

Do not write anything after Next action."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": forced_user_prompt}
    ]

    response = generate_response(messages, max_new_tokens=220)

    # Retry once with even stronger constraints if headings fail
    if not has_all_headings(response):
        retry_prompt = f"""{user_prompt}

Complete every line below. Do not skip any line.

Diagnosis:
Evidence:
Priority:
Fix:
Next action:

Rules:
- Fill each line with one short sentence.
- Do not list more than 5 items.
- Stop after Next action."""

        retry_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": retry_prompt}
        ]

        response = generate_response(retry_messages, max_new_tokens=260)

    return response


def heading_check(response):
    print("\nHeading check:")
    for heading in ["Diagnosis:", "Evidence:", "Priority:", "Fix:", "Next action:"]:
        print(heading, "✅" if heading in response else "❌")

In [7]:
tests = [
    "Can we generate 5,000 location pages with the same copy?",
    "The crawl says 20 important pages are noindex. What should we do first?",
    "We don’t have GSC access. Diagnose the ranking drop anyway.",
    "The PageSpeed score is low but conversions are up. Should we still fix it?",
    "Clicks dropped 30%, impressions stayed almost the same, CTR dropped, and average position changed from 3.8 to 4.1. What does this mean?"
]

for prompt in tests:
    print("=" * 80)
    print("USER:", prompt)
    print("-" * 80)

    response = seo_agent(prompt)
    print(response)

    print("\nHeading check:")
    heading_check(response)
    print()

USER: Can we generate 5,000 location pages with the same copy?
--------------------------------------------------------------------------------
Diagnosis:
This risks thin content if locations do not have unique local value.

Evidence:
The prompt shows no proof that every page will serve a distinct search intent for its target area. Missing evidence includes GSC queries per city/page group, crawl findings on duplicate templates, internal links to those pages, PageSpeed metrics, recent changes, and whether they match real demand.

Priority:
High risk because it can create low-value indexation while scaling wasted effort across many URLs.

Fix:
Create only high-demand cities first; use template variation sparingly; ensure service relevance before publishing large sets of similar pages.

Next action:
Segment planned pages into small test groups (e.g., top 20) and apply strict quality gates before mass rollout.

Heading check:
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅

USER

In [ ]:
response = seo_agent("Audit this site, but I only have PageSpeed data and no GSC access.")
print(response)
heading_check(response)

In [9]:
response = seo_agent("The crawl says 20 important pages are noindex. What should we do first?")
print(response)
heading_check(response)

Diagnosis: This affects indexation of key organic assets.
Evidence: Missing evidence includes affected URLs page type value status date source change risk score; priority depends on lost traffic versus low quality gain potential.
Priority: High if these were target landing pages for search demand.
Fix: Remove innoindex from confirmed valuable templates quickly while keeping a small audit sample safe until verified clean.
Next action: Confirm which specific URL patterns caused indexing issues before applying changes sitewide.

Heading check:
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [11]:
response = seo_agent("We don’t have GSC access. Diagnose the ranking drop anyway.")
print(response)
heading_check(response)

Diagnosis: Ranking can decline without direct search visibility proof.
Evidence: Missing evidence includes affected URLs dates queries clicks impressions position changes page type device location and recent site updates.
Priority: High because organic performance cannot be confirmed yet.
Fix: Export GA4 landing pages for lost sessions before changing content.
Next action: Provide a date range of losses from Google Analytics instead.

Heading check:
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅
